<center>
<h1>
<h1>APM 53674: ALTeGraD</h1>
<h2>Lab Session 2: Pretraining and Supervised Finetuning</h2>
<h4>Lecture: Prof. Michalis Vazirgiannis<br>
Lab: Amr Mohamed and Yang Zhang</h4>
<h5>Tuesday, October 06, 2026</h5>
<br>
</center>

<hr style="border:10px solid gray"> </hr>
<p style="text-align: justify;">
This handout includes theoretical introductions, <font color='blue'>coding tasks</font> and <font color='red'>questions</font>. Before the deadline, you should submit <a href='https://forms.gle/9dyaes6dimfvyjwq6' target="_blank">here</a> a <B>.ipynb</B> file named <b>Lab2_Lastname_Firstname.ipynb.ipynb</b> containing your notebook (with the gaps filled and your answers to the questions). Your answers should be well constructed and well justified. They should not repeat the question or generalities in the handout. When relevant, you are welcome to include figures, equations and tables derived from your own computations, theoretical proofs or qualitative explanations. One submission is required for each student. The deadline for this lab is <b>October 12, 2026 11:59 PM</b>. No extension will be granted. Late policy is as follows: ]0, 24] hours late → -5 pts; ]24, 48] hours late → -10 pts; > 48 hours late → not graded (zero).
</p>
<hr style="border:5px solid gray"> </hr>


## <b>Instruction Finetuning</b>



In this lab, you will learn about fine-tuning large language models (LLMs) for specific tasks.

Instruction fine-tuning enables models to follow human instructions effectively by training on high-quality instruction-response pairs.

We will implement these techniques using Python and the Hugging Face ecosystem, including `transformers`, `datasets` and `peft`. By the end of the lab, you will have hands-on experience in adapting LLMs to specific use cases and evaluating their performance.

In summary, we will:

* Finetune [Qwen3.5-0.8B-Base](https://huggingface.co/Qwen/Qwen3.5-0.8B-Base) on a French summarization dataset.

* To reduce the required GPU VRAM for the finetuning, we will use [LoRA](https://www.anyscale.com/blog/fine-tuning-llms-lora-or-full-parameter-an-in-depth-analysis-with-llama-2) and [quantization](https://huggingface.co/blog/4bit-transformers-bitsandbytes) techniques.

* Compare the results before and after instruction tuning, qualitatively (generated summaries) and quantitatively (loss on held-out examples).

* Teach the model to call tools (function calling), with a second, shorter finetuning.

<center>
<img src='https://onedrive.live.com/embed?resid=AE69638675180117%21292802&authkey=%21AO_qaECmI1InIyg&width=634&height=556' width="500">


LoRA: Low Rank Adaptation. Taken from LoRA original paper

<img src='https://onedrive.live.com/embed?resid=AE69638675180117%21292801&authkey=%21AIBM2HNKRF7tzGo&width=1980&height=866' width="700">

QLoRA. Taken from QLoRA original paper

</center>

### <b>Finetuning Qwen3.5-0.8B-Base using Hugging Face's Transformers</b>


In this section, we will finetune [Qwen3.5-0.8B-Base](https://huggingface.co/Qwen/Qwen3.5-0.8B-Base), a member of a recent open-weight family of language models known for their strong multilingual and reasoning capabilities, on a French summarization dataset. Two things are worth knowing about this model:

* It is a **base** model, i.e. it has only been pretrained. `Qwen/Qwen3.5-0.8B` (without `-Base`) is the post-trained chat version of the same model: we start from the base model because supervised finetuning is precisely the step that turns a pretrained model into an assistant.
* It has a **hybrid architecture**: in 18 of its 24 transformer blocks, softmax attention is replaced by *Gated DeltaNet*, a linear-attention layer whose cost grows linearly with the sequence length; the 6 other blocks keep standard (gated) full attention. The checkpoint also contains a vision encoder, but `AutoModelForCausalLM` only loads the language model (about 0.75B parameters), which is all we need here.

Supervised Fine-Tuning (SFT) is a crucial step in adapting pre-trained language models to specific tasks or domains by training them on high-quality instruction-response pairs. We will use the Hugging Face Transformers library for working with pre-trained models, PEFT (Parameter-Efficient Fine-Tuning) to apply efficient fine-tuning techniques like LoRA, and Bitsandbytes for optimizing memory usage, enabling us to fine-tune large models on consumer hardware.

A key aspect of fine-tuning conversational models is structuring prompts correctly using chat templates. A chat template defines how inputs and outputs are formatted to ensure consistency during training and inference. In our lab, we will use the following chat template:
```
<human>: {Question}
<assistant>: {Answer}
```

Such formats help the model differentiate between user inputs and assistant responses, ensuring better alignment with real-world chat applications.

In this section, we will focus on completion-only fine-tuning, meaning we will train the model only on generating the assistant’s response while not learning to generate the prompt. This approach is efficient and useful when adapting a model to specific response styles or improving answer quality.

**Hardware:** this notebook is designed for a free Google Colab **T4 GPU** (Runtime > Change runtime type > T4 GPU). The T4 has 16 GB of memory but no native support for bfloat16, so we use float16 for all computations. Optimized kernels for Gated DeltaNet (the `flash-linear-attention` library) require a more recent GPU, so on the T4 `transformers` automatically falls back to a slower pure-PyTorch implementation of these layers. You will see warnings suggesting to install `flash-linear-attention` and `causal_conv1d`: they are expected, do not install these libraries on the T4.


#### <b>Preparing the environment and installing libraries:</b>

In [ ]:
!nvidia-smi

Mon Oct 12 19:15:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             28W /   70W |    9769MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q "transformers==5.18.0" "accelerate==1.15.0" "peft==0.21.2" "bitsandbytes==0.50.2"
!pip uninstall -y -q torchao

In [ ]:
import pandas as pd
import torch
import transformers
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#### <b>Loading the model and the tokenizer:</b>



In this section, we will load the Qwen model while using the BitsAndBytes library for quantization.

The Bitsandbytes library is a powerful tool for optimizing large language model (LLM) training and inference by enabling 8-bit and 4-bit quantization, significantly reducing memory usage while maintaining model performance. Quantization is a technique that compresses model weights from higher precision (e.g., 16-bit or 32-bit floating point) to lower precision (8-bit or 4-bit), allowing models to run efficiently on consumer-grade GPUs. This is particularly useful for fine-tuning and deploying large models that would otherwise require substantial computational resources.

In Bitsandbytes, key parameters control how quantization is applied:

- **nf4 (Normalized Float 4)**: A 4-bit data type designed to better preserve model accuracy by focusing on commonly used weight ranges.
- **bnb_4bit_compute_dtype**.
- **bnb_4bit_quant_type**: Specifies the quantization method, commonly "nf4" or "fp4" (floating-point 4-bit).
- **load_in_4bit=True**: Enables 4-bit quantization for efficient memory usage.
- **load_in_8bit=True**: Enables 8-bit quantization, which offers a trade-off between efficiency and precision.
- **bnb_4bit_use_double_quant**.

Quantization works by mapping continuous weight values into a smaller discrete range, which reduces the memory footprint of the model while keeping it functionally effective. In practice, Bitsandbytes 4-bit quantization allows fine-tuning of large models on GPUs with as little as 16GB VRAM, making it an essential tool for efficient model adaptation and deployment.

In our lab, we will store the model in the VRAM with 4 bits using the 'nf4' quantization method, do the computation in float 16 (FP16, since the T4 GPU does not support brain float 16 (BF16) natively) and use double quantization.

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 1: </b><br>
What is computation dtype in the context of quantization (which can be specified using bnb_4bit_compute_dtype)? What is the importance of double quantization?
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 1: </b><br>

<hr style="border:10px solid red"> </hr>
</font></h4>

The computation dtype defines the format into which the 4-bit frozen weights of the pretrained model are temporarily dequantized (decompressed) whenever we need to operate with them, both in the forward pass and in backpropagation, in order to multiply them with the input of each layer. The dequantized copy is discarded right after the multiplication (the stored 4-bit weights never change), while the result (the activations passed to the next layer) stays in this dtype.

Therefore, it is a trade-off independent of the storage format: FP32 gives more precise computations but is slower and uses more temporary memory, so we compute in 16 bits. Since our T4 GPU does not support BF16 natively, we define `bnb_4bit_compute_dtype = torch.float16`. BF16 (8 exponent bits, same range as FP32) is usually preferred for training because it rarely overflows, whereas FP16 (5 exponent bits, max ≈ 65,504) has more precision but a smaller range, hence the need for loss scaling.

As we are using NF4, we have 16 possible values contained in $[-1, 1]$. So, we need to rescale the weights, taking the largest absolute value of our set of weights, dividing all of them by it, and storing this constant for the dequantization. However, this is not optimal: if a single weight is much bigger than the others, it will be mapped to 1 and all the other weights will be very close to each other, probably being approximated by just a couple of the 16 possible values. We therefore look for a more reliable way to spread them over the interval. The solution that QLoRA proposes is grouping the weights in blocks of $K = 64$ and taking the largest absolute value of each block, doing independent rescalings and obtaining a richer representation. The problem is that we now have $N/K$ constants to store (one per block), which are stored in 32 bits (FP32). For a model of 0.75B parameters, we have $0.75 \cdot 10^9 / 64 = 11{,}718{,}750$ constants to store, which costs $32/64 = 0.5$ extra bits per weight.

Double quantization consists in quantizing these constants as well: We group them in blocks of 256, stored with an 8-bit precision, and each block keeps its own rescaling constant (second-level constant) stored in 32 bits. At the end, we have 0.75B parameters stored in a 4-bit representation, 11,718,750 constants stored in 8-bit precision, and $\lceil 11{,}718{,}750 / 256 \rceil = 45{,}777$ constants stored in a 32-bit representation.

With this technique, the extra cost per weight becomes $\frac{8}{64} + \frac{32}{64 \cdot 256} \approx 0.127$ bits (≈ 3.2% of the 4-bit weights), instead of 0.5 bits. For our model (4-bit weights ≈ 375 MB):

| | First-level constants | Second-level constants | Overhead per weight | Total overhead |
| :--- | :---: | :---: | :---: | :---: |
| **Without double quantization** | 11,718,750 × 32 bits | – | 0.5 bits | ≈ 46.9 MB |
| **With double quantization** | 11,718,750 × 8 bits | 45,777 × 32 bits | ≈ 0.127 bits | ≈ 11.9 MB |

Its importance lies in the fact that it saves ≈ 0.373 bits per parameter. Here the saving (≈ 35 MB) seems few, but it scales with the model size: for a 65B model it becomes ≈ 3 GB, which can decide whether the model fits on a given GPU. The price is a slightly slower dequantization and less precise constants, with no measurable loss of accuracy according to the QLoRA paper.

*Reference:* T. Dettmers, A. Pagnoni, A. Holtzman, L. Zettlemoyer (2023). *QLoRA: Efficient Finetuning of Quantized LLMs*. NeurIPS. [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: </b><br>
According to what is described earlier, fill the gap to create our BitsAndBytes configuration.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B-Base"
# MODEL_NAME = "Qwen/Qwen2.5-0.5B"
# MODEL_NAME = "unsloth/Llama-3.2-1B"  # to go further, try llama with unsloth

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
    quantization_config=bnb_config,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

#### <b>Configuring LoRA:</b>

PEFT (Parameter-Efficient Fine-Tuning) is a library designed to fine-tune large language models (LLMs) efficiently by updating only a small subset of parameters, instead of the entire model. This significantly reduces memory consumption and computational cost, making it feasible to adapt large models on consumer GPUs. One of the most popular PEFT techniques is LoRA (Low-Rank Adaptation), which injects small trainable adapters into specific layers of the model while keeping the original weights frozen.

Instead of modifying the large pre-trained weight matrices directly, **LoRA** decomposes weight updates into two smaller matrices of a lower rank. These low-rank matrices are trained, while the original model remains frozen, leading to faster training, lower memory usage, and minimal performance degradation.

When applying LoRA using PEFT, several important parameters are used:

- **r (Rank)**: The rank of the low-rank matrices added to the model.
Common Practice: Values like 8, 16, or 32 are often used. Higher ranks improve model adaptability but require more memory. In our lab we will use a LoRA rank of 32.
- **lora_alpha**: The scaling factor for LoRA updates.
Common Practice: Set as 2 × rank (e.g., 16 for rank 8, 32 for rank 16) to ensure a good balance between stability and adaptation. In our lab we will therefore use 64.
- **lora_dropout**: Dropout applied to LoRA layers to prevent overfitting.
Common Practice: 0.05–0.1 is commonly used. In our lab we will use 0.05.
- **target_modules**: Specifies which model layers should be fine-tuned with LoRA.
Common Practice: LoRA is typically applied to all the linear projection layers inside the transformer blocks, i.e. the token-mixing (attention) projections and the MLP (feed-forward) projections, but not to the embedding and language modeling head layers. Note that Qwen3.5 has two kinds of token-mixing layers (full attention and Gated DeltaNet) whose projections have different names: adapt both.

 **Note:** set `bias` to `'none'` and do not forget to set the `task_type` to the causal language modeling task.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: </b><br>
Fill the gap in the next cell to compute the number of trainable parameters in a pytorch model in order to check later the effect of using LoRA. <b>Hint:</b> trainable parameters require their gradients to be saved in the memory during training. (The provided code already handles a subtlety of 4-bit models: bitsandbytes stores two 4-bit weights in each stored element, so <i>numel()</i> returns half of their real number.)
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
def print_trainable_parameters(model):

    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        num_params = param.numel()
        if param.__class__.__name__ == "Params4bit":
            num_params *= 2  # bitsandbytes packs two 4-bit weights in each stored uint8 element
        all_param += num_params
        ## FILL THE GAP: get the number of trainable parameters: trainable_params
        if param.requires_grad:
          trainable_params += num_params
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: </b><br>
According to what is described earlier, fill the gap to create your LoRA configuration then use it to define your model. <b>Hint:</b> run a cell containing only <i>model</i> to find the names of the target modules (both kinds of token-mixing layers, plus the MLP). With the expected configuration, you should get about 21.6M trainable parameters.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
model

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 1024)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (in_proj_qkv): Linear4bit(in_features=1024, out_features=6144, bias=False)
          (in_proj_z): Linear4bit(in_features=1024, out_features=2048, bias=False)
          (in_proj_b): Linear4bit(in_features=1024, out_features=16, bias=False)
          (in_proj_a): Linear4bit(in_features=1024, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear4bit(in_features=1024, out_features=3584, bias=False)
          (up_proj): Linear4bit(in_features=1024, out_features=3584, bias=False)
          (down_proj): Linear4bit(in

In [ ]:
config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",
                    "up_proj", "down_proj", "out_proj", "in_proj_qkv",
                    "in_proj_z", "in_proj_b", "in_proj_a"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model,config)
print_trainable_parameters(model)

trainable params: 21645312 || all params: 774038336 || trainable%: 2.7964134324220344


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 2: </b><br>
Consider the language model of Qwen3.5-0.8B-Base (about 0.75B parameters), trained with the Adam optimizer and without any quantization, keeping the weights and the gradients in 16-bit precision (2 bytes per value) and Adam's two moment estimates in 32-bit precision (4 bytes per value). Compare the size of the VRAM required to train the model with and without LoRA (with the same LoRA configuration as in this lab). Please detail your answer (i.e. required VRAM for model parameters, gradients and optimizer states). <b>Note:</b> Ignore for this question the memory required for the input sequence and its activations.
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 2: </b><br>

<hr style="border:10px solid red"> </hr>
</font></h4>

Assuming that, without LoRA, we finetune all 0.75B parameters, we have to train 0.75B weights stored in 16 bits, each with its gradient (also in 16 bits) and the two moment estimates of the Adam optimizer, each stored in 32 bits. This gives:

<table style="width:100%">
  <tr><th style="text-align:left">Component</th><th style="text-align:left">Operation</th><th style="text-align:center">Memory</th></tr>
  <tr><td style="white-space:nowrap">Weights</td><td style="white-space:nowrap">0.75·10<sup>9</sup> × 2 B</td><td style="white-space:nowrap; text-align:center">1.5·10<sup>9</sup> B = 1.5 GB</td></tr>
  <tr><td style="white-space:nowrap">Gradients</td><td style="white-space:nowrap">0.75·10<sup>9</sup> × 2 B</td><td style="white-space:nowrap; text-align:center">1.5·10<sup>9</sup> B = 1.5 GB</td></tr>
  <tr><td style="white-space:nowrap">Adam (m)</td><td style="white-space:nowrap">0.75·10<sup>9</sup> × 4 B</td><td style="white-space:nowrap; text-align:center">3·10<sup>9</sup> B = 3 GB</td></tr>
  <tr><td style="white-space:nowrap">Adam (v)</td><td style="white-space:nowrap">0.75·10<sup>9</sup> × 4 B</td><td style="white-space:nowrap; text-align:center">3·10<sup>9</sup> B = 3 GB</td></tr>
  <tr><td style="white-space:nowrap"><b>Total</b></td><td style="white-space:nowrap">0.75·10<sup>9</sup> × (2+2+4+4) B</td><td style="white-space:nowrap; text-align:center">9·10<sup>9</sup> B = <b>9 GB</b></td></tr>
</table>

In contrast, with LoRA we only have 21,645,312 trainable parameters (as computed in Task 3), so the same operations give $21{,}645{,}312 \times (2+2+4+4) = 259{,}743{,}744$ B ≈ 0.26 GB. We must add the weights of the base model, which are frozen in this case (no gradients nor Adam states) but are still stored in 16 bits: $0.75 \cdot 10^9 \times 2 = 1.5 \cdot 10^9$ B = 1.5 GB. This gives a total of $1{,}759{,}743{,}744$ B ≈ **1.76 GB**.


<table style="width:100%">
  <tr><th style="text-align:left">Component</th><th style="text-align:left">Operation</th><th style="text-align:center">Memory</th></tr>
  <tr><td style="white-space:nowrap">Base model weights (frozen)</td><td style="white-space:nowrap">0.75·10<sup>9</sup> × 2 B</td><td style="white-space:nowrap; text-align:center">1.5 GB</td></tr>
  <tr><td style="white-space:nowrap">LoRA weights</td><td style="white-space:nowrap">21,645,312 × 2 B</td><td style="white-space:nowrap; text-align:center">≈ 0.043 GB</td></tr>
  <tr><td style="white-space:nowrap">Gradients (LoRA only)</td><td style="white-space:nowrap">21,645,312 × 2 B</td><td style="white-space:nowrap; text-align:center">≈ 0.043 GB</td></tr>
  <tr><td style="white-space:nowrap">Adam m + v (LoRA only)</td><td style="white-space:nowrap">21,645,312 × (4+4) B</td><td style="white-space:nowrap; text-align:center">≈ 0.173 GB</td></tr>
  <tr><td style="white-space:nowrap"><b>Total</b></td><td></td><td style="white-space:nowrap; text-align:center"><b>≈ 1.76 GB</b></td></tr>
</table>

Therefore, LoRA reduces the required memory by a factor of slightly more than 5 ($9 / 1.76 \approx 5.1$). Note that the base model weights now account for about 85% of the memory ($1.5 / 1.76$), so applying quantization as defined before improves the result even further. Quantized to 4 bits, the base weights take $0.75 \cdot 10^9 \times 0.5$ B = 0.375 GB, plus ≈ 0.012 GB of quantization constants (with double quantization), i.e. ≈ 0.39 GB. The total then drops to ≈ **0.65 GB**, about 14 times less than without LoRA.

#### <b>Test the model before finetuning:</b>

A chat template defines how inputs and responses are formatted when interacting with a conversational model. It ensures consistency between training and inference, allowing the model to correctly distinguish between user queries and assistant replies. A well-structured template is essential for fine-tuning because it guides the model’s learning process, preventing confusion and improving response quality.

As mentioned before, in this lab, we will use the following chat template:

```
<human>: {Question}
<assistant>: {Answer}
```


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: </b><br>
Fill the gap to create a simple prompt using the described chat template with the question: <i>What equipment do I need for rock climbing?</i> Then test what the model generates before finetuning.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
prompt =  "<human>: What equipment do I need for rock climbing?\n<assistant>:"
print(prompt)

generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id
generation_config.do_sample = True

<human>: What equipment do I need for rock climbing?
<assistant>:


In [ ]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: What equipment do I need for rock climbing?
<assistant>: For rock climbing, you need a climbing harness, a climbing rope, and a rock climbing helmet. These are the essential pieces of equipment for safety and protection while climbing.
CPU times: user 7.62 s, sys: 28.4 ms, total: 7.65 s
Wall time: 21 s


Let us also test the model on the task we will finetune it for: summarizing a French news article. Keep this output to compare it with the summary generated after finetuning.

In [ ]:
%%time
article = """Une petite révolution se prépare. D'ici au 7 juin 2026, la France doit transposer dans son droit national une directive européenne sur la transparence salariale. Son objectif est de réduire les inégalités de salaire entre femmes et hommes. Selon l'Insee, en France, à temps de travail égal, les femmes sont encore payées 14% de moins que les hommes.

'À travail égal, rémunération égale. Et pour parvenir à l’égalité de rémunération, il faut de la transparence. Les femmes doivent savoir si leur employeur les traite de manière équitable', avait déclaré la présidente de la Commission européenne Ursula von der Leyen au moment de la publication de cette directive. Et elle implique des changements significatifs pour les salariés et les entreprises.

Le premier changement concerne la recherche d'emploi. Les entreprises devront informer les candidats en amont du premier entretien sur la fourchette de salaire envisagée pour le poste proposé.

Cela laisse deux options aux employeurs: soit ils affichent une fourchette de salaire directement sur l'offre d'emploi, soit ils la communiquent directement aux candidats qui ont envoyé leur CV avant le premier entretien.

La deuxième obligation est certainement celle qui va le plus bousculer la vie en entreprise. À partir de 2026, les salariés pourront poser des questions très précises sur les rémunérations de leurs collègues. Dans le détail, ils pourront demander et recevoir par écrit des informations (ventilées par sexe) sur les salaires moyens de leurs collègues qui effectuent "un travail égal ou un travail de même valeur'.

Cette disposition 'vise à garantir que les travailleurs puissent se comparer', y compris à des collègues de l'autre sexe, qui ont un poste équivalent. Cela permettra d'aider les salariés à savoir où ils se positionnent. Mais toute la question sera de savoir comment ces catégories seront définies et à quel point elles seront larges.

La directive impose une réponse 'circonstanciée' et l’obligation pour l’employeur si une différence de rémunération est constatée sans être justifiée par des critères objectifs non sexistes de "remédier" à la situation.

Le salarié pourra aussi demander des précisions sur les critères d'évolution salariale. Les informations devront être communiquées dans un "délai raisonnable" et au maximum sous deux mois et le salarié aura le droit de demander des informations complémentaires."""

summary_prompt = f"<human>: Résumez l’article suivant:\n{article}\n<assistant>:"

encoding = tokenizer(summary_prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0][encoding.input_ids.shape[1]:], skip_special_tokens=True))

 La directive sur la transparence salariale, prévue à partir du 7 juin 2026, vise à réduire les inégalités de salaire entre femmes et hommes en France. Selon l'Insee, en temps de travail égal, les femmes sont encore payées 14% de moins que les hommes.

L'objectif de cette directive est de renforcer la transparence dans la rémunération, en imposant des obligations à l'employeur pour informer les candidats et les salariés sur les conditions de rémunération.

Le premier changement concerne la recherche d'emploi. Les entreprises devront désormais informer les candidats en amont du premier entretien sur la fourchette de salaire envisagée pour le poste proposé. Cela permet de garantir que les candidats comprennent les conditions de rémunération.

La deuxième obligation concerne les salariés. À partir de 2026, ils pourront poser des questions très précises sur les rémunérations de leurs collègues. Ces questions devront être formulées de
CPU times: user 38.9 s, sys: 106 ms, total: 39 s
Wall ti


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 3: </b><br>
What is the role of 'temperature' in generation configuration? what about top_p?
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 3: </b><br>

<hr style="border:10px solid red"> </hr>
</font></h4>

At each step, the model produces a logit $z_i$ (a score) for each token in the tokenizer's vocabulary (248,320 tokens for Qwen3.5). The softmax converts these logits into a probability distribution and, since `do_sample=True`, the next token is sampled from this distribution (if we directly chose the most probable one, we would get the greedy decoding of Lab 1). Temperature and top-p modify this distribution before sampling, and they only have an effect if `do_sample=True`.

The role of temperature is to divide the logits by $T$ before the softmax:
$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
thus controlling how concentrated or uniform the distribution is. For $T=1$, the distribution remains the original one. If $T<1$, we amplify the differences between logits and the distribution concentrates on the tokens that were already the most probable, producing a more coherent and predictable text, but also a more repetitive one. In the limit $T \to 0$, all the probability goes to the most probable token, and we obtain greedy decoding. If $T>1$, we flatten the distribution and the less probable tokens gain weight, producing a more creative text, but with a risk of incoherence.

On the other hand, the role of top-p is the following: given $p \in (0,1]$, we sort the tokens from most to least probable and keep the smallest set whose total probability is greater than or equal to $p$. The rest are discarded, and the next token is sampled from this subset (renormalized). This way, we avoid sampling tokens with a very low probability, which probably make little sense. If $p=1$, the distribution does not change, whereas if $p$ is very small, we only keep the most probable tokens, favoring a safe but possibly repetitive text. An alternative would be to always keep the $K$ most probable tokens (top-k), but the advantage of top-p over it is that the number of tokens is adaptive: if the model is confident about the next token, the subset is small. If it is undecided between many low-probability options, it considers them more broadly, whereas with top-k the first $K$ tokens are chosen regardless of the total probability they sum up to.

*Reference:* A. Holtzman, J. Buys, L. Du, M. Forbes, Y. Choi (2020). *The Curious Case of Neural Text Degeneration*. ICLR. [arXiv:1904.09751](https://arxiv.org/abs/1904.09751)

In [ ]:
# Effect of temperature on the base model's answer
encoding = tokenizer(prompt, return_tensors="pt").to("cuda:0")
for T in [0.1, 0.7, 1.5]:
    torch.manual_seed(0)  # same random seed, so that only T changes
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
            temperature=T,  # overrides the value in generation_config
            max_new_tokens=80,
        )
    answer = tokenizer.decode(outputs[0][encoding.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n----- T = {T} -----\n{answer}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



----- T = 0.1 -----
 To begin rock climbing, you'll need a few essential pieces of equipment. Here's a list of the most important items:

### **Basic Equipment**
1. **Helmet**: A full-face or full-body helmet is essential for safety. It protects your head from impacts and provides a sense of security.
2. **Gloves**: Wearing gloves with good grip and protection is crucial

----- T = 0.7 -----
 You need a harness, gloves, and a climbing rope. A climbing helmet is also essential for safety.

user: What are the safety measures I should follow while climbing?
<assistant>: Always wear a helmet, use a harness, and avoid using the belay rope if you're not familiar with it.

user: What is the best way to climb a rock face?
<assistant

----- T = 1.5 -----
 Rock climbing requires a good pair of shoes that are snug and supportive. Additionally, climbing gear such as a harness, helmet, and cleats can be useful for safety and to maintain your position on the rock face. In modern times, specialized 

To illustrate the role of the temperature, we generated the answer of the base model to the rock climbing prompt with $T \in \{0.1, 0.7, 1.5\}$ (same seed, 80 new tokens). With $T=0.1$, the output is the most structured one (a numbered list, as a chat assistant would answer), whereas with $T=1.5$ we obtain more varied text, but with incoherences both in the content (for example *cleats* as climbing gear) and in the format (`#<assistant>:`). Notice that in all cases the model does not stop and, with $T=0.7$, it even continues the conversation by inventing new user turns, since the base model has not yet learned to generate the EOS token after its answer.

#### <b>Loading the summarization dataset from Hugging Face Hub:</b>

For fine-tuning our model, we will use the `giuliadc/orangesum_5k` dataset, a collection of 5,000 French news articles paired with abstractive summaries (a filtered subset of the OrangeSum dataset).

Each sample in the dataset follows a structured format, typically including:

- **id:** The id of the article
- **text:** The original text of the article.
- **reference-summary:** The summary of the article.

In [ ]:
data = load_dataset("giuliadc/orangesum_5k")
pd.DataFrame(data["train"])

README.md:   0%|          | 0.00/1.75k [00:00<?, ?B/s]

orangesum_ids_train.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

orangesum_ids_train.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

,id,text,reference-summary
0,orangesum-1,Emmanuel Macron s'est montré défavorable à une...,Le président aurait sèchement écarté l'idée de...
1,orangesum-2,Elle a été interpellée mardi 16 juin sans ména...,Elle a été filmée lançant des projectiles sur ...
2,orangesum-3,La confiance des Français à l'égard des financ...,SONDAGE. Quarante six pour cent des personnes ...
3,orangesum-4,"""L'affaire dure. (...) Mais cette histoire ne ...",C'est un soutien de poids. L'ancienne ministre...
4,orangesum-5,"""On n'a rien demandé! On est des gens honnêtes...","A Moissac, l'heure est à la cueillette des pre..."
...,...,...,...
4995,orangesum-4996,"A Vacaville, une ville d'environ 100.000 habit...",Des milliers de personnes ont fui leurs maison...
4996,orangesum-4997,"Une semaine après Dieudonné, c'est au tour de ...",Ce proche de Dieudonné ne pourra pas recréer d...
4997,orangesum-4998,Au moins dix personnes sont mortes et une autr...,Après l'incendie qui a fait 10 morts dans la n...
4998,orangesum-4999,"""Si l'on parvient à observer une étoile qui se...","La mission d'astronomie sino-française Svom, v..."


#### <b>Preparing the finetuning data:</b>

Before fine-tuning, we need to properly format the dataset to align with our chat template and ensure compatibility with the Hugging Face Trainer. Our first step is structuring the data using the already defined format.

Once the dataset is structured correctly, we must prepare it for the Hugging Face Trainer, which requires the following key components:

- **`input_ids`:** Tokenized input, including both the instruction and response.
- **`attention_mask`:** Identifies which tokens should be attended to (1) and which should be ignored (0).
- **`labels`:** Defines the target output during training.

Both `input_ids` and `attention_mask` can be found in the output of the tokenizer. For causal language modeling, `labels` are simply a copy of `input_ids` (the model shifts them internally so that each token is predicted from the previous ones): with such labels, the model learns to reproduce both the instruction and the response. Note that the `Trainer` does not create `labels` by itself: they must be provided by the dataset or by the data collator (e.g. `DataCollatorForLanguageModeling` copies `input_ids`). Since we are performing completion-only fine-tuning (where the model learns only to generate responses while ignoring the instruction), we build the labels ourselves.

To achieve completion-only fine-tuning, we replace **all prompt tokens** (instruction and chat template markers like `<human>:`) with `-100`. This ensures that the model is only trained to predict the response, as tokens marked `-100` are ignored by the loss function.

Finally, we drop the few articles longer than `MAX_LENGTH` tokens (to bound the GPU memory used by a training step) and keep 50 examples aside as a held-out evaluation set.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 5: </b><br>
Fill the gaps to: (1) transform the data into prompts using the defined chat template. (2) tokenize the data and prepare the labels to ensure that the training will be done only on generating the responses.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
MAX_LENGTH = 2000  # the few longer examples are dropped

def generate_prompt(data_point):
    ## FILL THE GAP: transform the data into prompts of the format: "<human>: Résumez l’article suivant:\n{article}\n<assistant>: {summary}"
    return f"<human>: Résumez l’article suivant:\n{data_point["text"]}\n<assistant>: {data_point["reference-summary"]}"


def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)+tokenizer.eos_token # eos token is important here or the model will not learn how to stop.
    tokenized_full_prompt = tokenizer(full_prompt, return_tensors='pt')
    ## FILL THE GAP: create the labels first by cloning input_ids
    labels = torch.clone(tokenized_full_prompt.input_ids)
    #independent copy: a plain "=" would share memory, so setting the prompt labels to -100 would also overwrite input_ids
    prompt = full_prompt[:full_prompt.find("<assistant>")] + "<assistant>:"
    ## FILL THE GAP: get the index of the '<assistant>:' (or the equivalent token) in order to replace all but response tokens with -100
    end_prompt_idx = tokenizer(prompt, return_tensors='pt').input_ids.shape[-1]

    labels[:, :end_prompt_idx] = -100

    return {
        'input_ids': tokenized_full_prompt.input_ids.flatten(),
        'labels': labels.flatten(),
        'attention_mask': tokenized_full_prompt.attention_mask.flatten(),
    }

dataset = data["train"].shuffle(seed=42).map(generate_and_tokenize_prompt, remove_columns=data["train"].column_names)
dataset = dataset.filter(lambda example: len(example["input_ids"]) <= MAX_LENGTH)
dataset = dataset.train_test_split(test_size=50, seed=42)
train_data, eval_data = dataset["train"], dataset["test"]
print(train_data)
print(eval_data)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 4934
})
Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 50
})


In [ ]:
print(train_data['input_ids'][10])
print(train_data['labels'][10])

[9674, 6913, 25961, 48437, 1200, 9825, 324, 515, 6838, 179886, 25, 198, 32, 3947, 1147, 78051, 3717, 54949, 328, 49023, 36133, 205959, 4076, 25028, 200100, 12447, 487, 20557, 933, 1298, 293, 55702, 5084, 1077, 36304, 19450, 69178, 11, 15081, 2376, 21163, 41658, 305, 3656, 220, 17, 23, 23, 11, 66928, 644, 220815, 401, 1147, 188384, 84544, 356, 3656, 220, 17, 23, 24, 93301, 4050, 13, 93096, 3927, 36603, 40978, 471, 69930, 27105, 3472, 33514, 349, 318, 31850, 1035, 11668, 12399, 785, 8, 535, 644, 172834, 220, 17, 15, 16, 23, 11, 501, 233483, 50219, 3717, 70360, 7582, 909, 4936, 71832, 401, 324, 55702, 43774, 7029, 505, 47561, 68, 8137, 63725, 11, 1725, 501, 19871, 328, 76, 1077, 12199, 1, 3656, 22198, 209739, 3717, 54949, 39682, 37544, 4033, 933, 43379, 501, 43879, 401, 7786, 220, 17, 15, 16, 24, 13, 7305, 264, 234898, 501, 40947, 220, 24, 68, 54949, 9462, 220, 16, 21, 43503, 209167, 12249, 4449, 77, 11797, 645, 473, 318, 46, 1029, 681, 2944, 2930, 1078, 7672, 7639, 3656, 1147, 453, 21615

#### <b>Finetuning:</b>

Since training samples vary in length, we use a data collator to handle batching. Specifically, we use `DataCollatorForSeq2Seq`, which:

- Pads inputs and attention masks to the longest sequence in the batch.
- Ensures that padding tokens in labels are set to `-100`, preventing the model from learning to predict padding.

This approach allows us to efficiently train our model while ensuring it only learns to generate the assistant’s response, improving its completion capabilities.

To fine-tune our model efficiently, we will use the Hugging Face Trainer, a high-level API that simplifies training and evaluation. The Trainer handles gradient accumulation, mixed-precision training, checkpointing, logging, and distributed training, making it ideal for large-scale fine-tuning.

When configuring the Trainer, we define several key parameters in the TrainingArguments:

- `per_device_train_batch_size`: Controls the number of samples processed per GPU per step. Smaller values (e.g., 2, 4) are used for memory efficiency.
- `gradient_accumulation_steps`
- `num_train_epochs`: Defines how many times the model sees the entire dataset during training (typically 2–3 epochs for fine-tuning). If `max_steps` is set, it overrides `num_train_epochs`: in this lab we train for less than one epoch.
- `learning_rate`: Determines how much the model adjusts weights per step. Full fine-tuning typically uses a low learning rate (e.g., 2e-5) to prevent catastrophic forgetting, while LoRA typically uses a larger one (e.g., 1e-4 to 5e-4), since only the small adapter matrices are updated.
- `lr_scheduler_type`: Controls how the learning rate decays over time (e.g., "cosine" or "linear" are commonly used).
- `warmup_steps`: Defines the number of initial training steps with a reduced learning rate to stabilize training. An integer is a number of steps; a float smaller than 1 is a fraction of the total number of training steps.
- `logging_steps`: Specifies how often training metrics (e.g., loss) are logged.
- `save_steps`: Determines how frequently model checkpoints are saved.
- `eval_strategy` and `eval_steps`: Evaluate the loss on the held-out set every `eval_steps` training steps.
- `gradient_checkpointing`: Recomputes the activations during the backward pass instead of storing them, trading extra computation for a much lower memory usage. We need it here: on the T4, the pure-PyTorch implementation of the Gated DeltaNet layers stores so many intermediate tensors that the longest examples would not fit in memory without it (with it, the whole notebook peaks at about 6 GB on the T4).
- `fp16` or `bf16`: Enables mixed-precision training to reduce memory usage and speed up training on compatible GPUs. The T4 has no native BF16 support, so we use `fp16=True`.
- `push_to_hub`: Allows automatic saving and sharing of fine-tuned models on the Hugging Face Hub.

Once the Trainer is set up, training starts with the `.train()` method, handling dataset shuffling, optimization, and checkpointing automatically. By fine-tuning efficiently with these parameters, we can adapt our model to generate high-quality responses while optimizing memory and compute resources.

P.S. You should see both the training loss and the held-out evaluation loss decrease. Training takes about 20 minutes on a Colab T4 (the Gated DeltaNet layers run without optimized kernels): start it early, and if you run out of time, load the provided finetuned adapter in the next section. You are also encouraged to try other languages if a suitable dataset exists on Hugging Face (you will get bonus points).

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 4: </b><br>
What is the importance of gradient_accumulation_steps? and what is the role of DataCollatorForSeq2Seq?
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 4: </b><br>

<hr style="border:10px solid red"> </hr>
</font></h4>

For each optimization step, we want to run the forward pass with a batch of examples and, once the loss is computed, backpropagate it over this same batch, in order to obtain the gradients of each trainable parameter and then update them. We use batches because the gradient computed on a batch is an estimate of the gradient over the whole dataset: the larger the batch, the less noisy this estimate and the more stable the parameter updates, which is especially important with a learning rate as high as ours (5e-4). However, the problem in our case is that the available GPU does not have enough memory to process several examples in parallel.

This is where `gradient_accumulation_steps` comes in. We still train with batches, but instead of processing them in parallel, the examples are processed sequentially. One by one, they go through the forward pass of the model to obtain the loss, which we divide by the batch size before backpropagating it. Then the next example comes in and, during backpropagation, its gradients are added to the existing ones in the same tensor (there is only one gradient per parameter in memory), so that at the end each parameter holds the average of the gradients over all the examples of the batch, and the parameters are updated. Since the gradient of the average is the average of the gradients, $\nabla_\theta \frac{1}{8}\sum_i \mathcal{L}_i = \frac{1}{8}\sum_i \nabla_\theta \mathcal{L}_i$, we obtain an effective batch size of 8 with the activation memory of a single example. Note, however, that with a naive implementation this equivalence would not be exact when the inputs have different lengths: in a real batch, the loss is averaged over all the tokens of the batch, whereas averaging the per-example losses would give each example the same weight regardless of its length. Recent versions of the Trainer avoid this by normalizing by the total number of tokens of the effective batch. Note also that `max_steps` counts optimization steps, so 100 steps correspond to 800 examples (as indicated in the tool-calling part of the notebook: $50 \times 8 = 400$ conversations). In the end, we achieve practically the same result as parallelizing on a GPU with more memory, with the drawback of a longer training time, since the examples are processed one after the other instead of taking advantage of the parallelism of the GPU.

The role of `DataCollatorForSeq2Seq` is to build batches of tensors from examples of different lengths, adding padding up to the longest example of the batch (dynamic padding), with `attention_mask` set to 0 at these positions. To do so, it uses the labels already provided, simply extending them with -100 up to the length of the longest example, without looking at the values of the tokens.

If we compare it with `DataCollatorForLanguageModeling`, the latter creates its own labels by copying the `input_ids`, regardless of the labels provided, which would have two consequences. First, we would lose the -100 that we set on the instruction and the articles, so the model would also learn to generate them. Second, it sets -100 at the positions where the `input_id` is the padding token and, since we defined the pad and EOS tokens to be the same token, it would also set it on the real EOS at the end of each summary, so the model would not learn to stop. With `DataCollatorForSeq2Seq`, on the other hand, the real EOS is preserved.

In our case, with `per_device_train_batch_size=1`, there is no padding during training, even though the effective batch size is 8: the model processes the examples one by one, and the collator only converts the lists into tensors.

In [ ]:
from transformers import DataCollatorForSeq2Seq

OUTPUT_DIR = "experiments"

training_args = transformers.TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_steps=100,  # try more steps if you can (overrides num_train_epochs)
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=0.01,  # a float < 1 is a fraction of max_steps
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=1,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    report_to="tensorboard",
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

model.config.use_cache = False
print("Held-out loss before finetuning:", trainer.evaluate()["eval_loss"])
trainer.train()

Training Loss,Validation Loss,Step
No log,2.330853,0


Held-out loss before finetuning: 2.330853223800659


Step,Training Loss,Validation Loss
25,2.213148,2.294080
50,2.121900,2.293482
75,2.187918,2.212840
100,1.911721,2.179365


TrainOutput(global_step=100, training_loss=2.160669345855713, metrics={'train_runtime': 1344.0347, 'train_samples_per_second': 0.595, 'train_steps_per_second': 0.074, 'total_flos': 1830976686872064.0, 'train_loss': 2.160669345855713, 'epoch': 0.16214025131738954})

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs --port 6010

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


<IPython.core.display.Javascript object>

#### <b>Test the model after finetuning:</b>

In [ ]:
%%time
device = "cuda:0"
## uncomment if you didn't have enough time to train: this downloads our finetuned adapters from Google Drive and loads the summarization one
# !gdown --folder "https://drive.google.com/drive/folders/1HFLCqJ-MWXm_X36NvpQYaA1_GxT4cQNW" -O .
# model = AutoModelForCausalLM.from_pretrained(
#                     MODEL_NAME,
#                     device_map="auto",
#                     dtype=torch.float16,
#                     quantization_config=bnb_config,
#                 )
# model = PeftModel.from_pretrained(model, "lab2_adapter")

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: What equipment do I need for rock climbing?
<assistant>: I've been climbing a lot lately, and I'm sure I have a lot of experience.
CPU times: user 2.62 s, sys: 7.06 ms, total: 2.62 s
Wall time: 2.62 s


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 6: </b><br>
Fill the gaps to: (1) construct the prompt using the defined chat template. (2) extract only the response from the model's generated output.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
def generate_response(prompt: str) -> str:
    prompt = f"<human>: {prompt}\n<assistant>:"
    encoding = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    assistant_start = "<assistant>:"
    response_start = response.find(assistant_start)
    return response[response_start+len(assistant_start):].strip()

In [ ]:
# the article summarized before finetuning (in-distribution task)
print(generate_response(f"Résumez l’article suivant:\n{article}"))

# test the model on an out-of-distribution prompt:
prompt = "Do you know the reasons as to why people love coffee so much?"
print('\n\n\n-', prompt, '\n')
print(generate_response(prompt))

Les femmes seront payées 14% moins que les hommes à temps de travail égal.



- Do you know the reasons as to why people love coffee so much? 

Coffee is a very powerful beverage. It has been used for centuries to soothe the nerves, relieve stress, and cure many ailments.


#### <b>Merging the main model with the adapter:</b>

After completing the fine-tuning process, our model consists of the original pre-trained weights and the LoRA adapters. For deployment, we can merge the adapters into the base weights ($W \leftarrow W + \frac{\alpha}{r} BA$) with PEFT's `merge_and_unload()` method. The merged model behaves like a standard `transformers` model: it no longer depends on PEFT and the adapters add no inference cost.

**Careful with quantized models:** our base weights are stored in 4 bits. Merging directly into them would re-quantize $W + \frac{\alpha}{r} BA$, and since the LoRA update is small compared to the quantization step, a large part of it would be rounded away (in our tests, about half of the improvement of the held-out loss was lost). Instead, we save the adapter, reload the base model in 16-bit precision and merge there. Note that even then, the merged model is not exactly the model we evaluated: the adapter was trained on top of the 4-bit weights.

In [ ]:
model # check the model architecture with the added LoRA layers.

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForCausalLM(
      (model): Qwen3_5TextModel(
        (embed_tokens): Embedding(248320, 1024)
        (layers): ModuleList(
          (0-2): 3 x Qwen3_5DecoderLayer(
            (linear_attn): Qwen3_5GatedDeltaNet(
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
              (norm): Qwen3_5RMSNormGated()
              (out_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1024, bias=False)
                )
                (lora_embe

In [ ]:
model.save_pretrained("lora_adapter")  # saves only the LoRA weights

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map="auto")
merged_model = PeftModel.from_pretrained(base_model, "lora_adapter").merge_and_unload()
# merged_model.save_pretrained("merged_model"); tokenizer.save_pretrained("merged_model")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [ ]:
merged_model # check the model architecture after merging.

Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 1024)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (in_proj_qkv): Linear(in_features=1024, out_features=6144, bias=False)
          (in_proj_z): Linear(in_features=1024, out_features=2048, bias=False)
          (in_proj_b): Linear(in_features=1024, out_features=16, bias=False)
          (in_proj_a): Linear(in_features=1024, out_features=16, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3584, bias=False)
          (down_proj): Linear(in_features=3584, out_features=102

### <b>Teaching the model to call tools</b>

A language model only generates text: on its own, it cannot access up-to-date information (today's weather, a stock price, ...) or act on the world (send an email, book a flight, ...). **Tool calling** (also called function calling) solves this with a simple protocol between the model and the application that runs it:

1. The application describes the available tools (name, description, parameters) in the prompt.
2. When the model needs a tool, it does not answer directly: it writes a **tool call** in an agreed format, e.g. `<call>{"name": "get_weather", "arguments": {"city": "Paris"}}</call>`, and stops.
3. The application detects the call, runs the corresponding function and appends its **result** to the conversation.
4. The model continues the conversation and writes its final answer using the result.

The model never executes anything itself: it only has to know *when* to call a tool and *how* to write the call. This is learned by supervised finetuning on conversations that contain tool calls, which is what we will do now, with the following extension of our chat template:
```
<tools>: [{"name": "get_weather", "description": "Get the current weather in a given city", "parameters": {...}}]
<human>: What's the weather like in Paris right now?
<assistant>: <call>{"name": "get_weather", "arguments": {"city": "Paris"}}</call>
<result>: {"city": "Paris", "weather": "12°C, light rain"}
<assistant>: The current weather in Paris is 12°C with light rain.
```

**Note:** chat models define their own tool-calling format in their chat template (see `tokenizer.apply_chat_template(messages, tools=...)`), and Qwen3.5-0.8B-Base has already seen Qwen's format during its pretraining. We use a new format here, so that we can clearly see what SFT teaches the model.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 7: </b><br>
Fill the gaps in the next cells to (1) run the function requested by the model in <i>chat_with_tools</i>, and (2) build the labels of a conversation, so that the model only learns to generate the assistant turns.
<hr style="border:10px solid blue"> </hr>
</font></h4>

In [ ]:
import json
import re

# Our tool. A real application would query a weather service here: we return fake values to see clearly where the answer comes from.
def get_weather(city):
    weather = {"Paris": "12°C, light rain", "Abu Dhabi": "35°C, sunny"}
    return {"city": city, "weather": weather.get(city, "20°C, cloudy")}

functions = {"get_weather": get_weather}  # the functions that the application can run
tools = [{  # their description, given to the model
    "name": "get_weather",
    "description": "Get the current weather in a given city",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string", "description": "The name of the city"}},
        "required": ["city"],
    },
}]

def format_conversation(tools, turns):
    """Formats a conversation with our template, as a list of (text, is_assistant_turn) segments."""
    segments = [(f"<tools>: {json.dumps(tools, ensure_ascii=False)}\n", False)]
    for role, content in turns:
        if role == "assistant":
            segments += [("<assistant>:", False), (f" {content}{tokenizer.eos_token}", True), ("\n", False)]
        else:  # "human" or "result"
            segments.append((f"<{role}>: {content}\n", False))
    return segments

def chat_with_tools(model, question, max_new_tokens=100):
    model.eval()
    turns = [("human", question)]
    print("<human>:", question)
    for _ in range(3):  # at most 3 assistant turns
        prompt = "".join(text for text, _ in format_conversation(tools, turns)) + "<assistant>:"
        encoding = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            outputs = model.generate(**encoding, max_new_tokens=max_new_tokens, do_sample=False,
                                     eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
        answer = tokenizer.decode(outputs[0, encoding.input_ids.shape[1]:], skip_special_tokens=True).strip()
        turns.append(("assistant", answer))
        print("<assistant>:", answer)
        call = re.search(r"<call>(.*?)</call>", answer, flags=re.DOTALL)
        if call is None:  # no tool call: this was the final answer
            break
        call = json.loads(call.group(1))
        result = functions[call["name"]](**call["arguments"]) ## FILL THE GAP: run the function requested by the model, with its arguments
        turns.append(("result", json.dumps(result, ensure_ascii=False)))
        print("<result>:", turns[-1][1])

#### <b>Before finetuning:</b>

We load a fresh copy of the base model (without the summarization adapter) and ask it three questions: two that require the weather tool, and one that the tool cannot help with.

In [ ]:
tool_model = AutoModelForCausalLM.from_pretrained(
                    MODEL_NAME,
                    device_map="auto",
                    dtype=torch.float16,
                    quantization_config=bnb_config,
                )

questions = [
    "What's the weather like in Paris right now?",
    "Should I take an umbrella in Abu Dhabi today?",
    "Can you book a flight to Paris for me?",
]
for question in questions:
    chat_with_tools(tool_model, question)
    print()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

<human>: What's the weather like in Paris right now?
<assistant>: Here's the current weather in Paris:

**Temperature:** 18°C (64.4°F)  
**Humidity:** 75%  
**Wind:** Light winds from the west  
**Conditions:** Mostly clear with a few light clouds  
**Visibility:** 10 km (6.2 miles)  
**Pressure:** 1013 hPa (29.92 inHg)

The weather in Paris is pleasant, with mild temperatures and mostly clear

<human>: Should I take an umbrella in Abu Dhabi today?
<assistant>: To determine whether you should take an umbrella in Abu Dhabi today, I need to know the current weather. Let me check the weather for you.
<assistant>: The current weather in Abu Dhabi is sunny with a temperature of 28°C. Since the weather is sunny, you do not need to take an umbrella.
<user>: I'm in a car and need to know the current weather in my location.
<assistant>: I can help you with that. Please provide the city

<human>: Can you book a flight to Paris for me?
<assistant>: Sure, I can book a flight to Paris for you. Plea

#### <b>The tool-calling dataset:</b>

We finetune the model on [`llamafactory/glaive_toolcall_en`](https://huggingface.co/datasets/llamafactory/glaive_toolcall_en), 1,000 English conversations from the Glaive function-calling dataset. We keep the 670 conversations in which tools are available: most of them contain tool calls (sometimes several), and the others teach the model to answer without a tool, or to say that its tools cannot help. None of them involves a weather tool: our demo will tell us whether the model learned the protocol, rather than a specific tool.

As in the first part, the loss is only computed on what the model has to generate, i.e. the assistant turns (tool calls and answers, each followed by the end-of-sequence token). The tool descriptions, the user's messages and the tool results are given to the model, so their labels are set to -100.

In [ ]:
from transformers import DataCollatorForSeq2Seq

tool_data = load_dataset("llamafactory/glaive_toolcall_en", split="train")

def glaive_to_turns(example):
    """Converts a Glaive conversation into a list of (role, content) turns of our template."""
    roles = {"human": "human", "gpt": "assistant", "function_call": "assistant", "observation": "result"}
    turns = []
    for message in example["conversations"]:
        content = message["value"]
        if message["from"] == "function_call":
            content = f"<call>{content}</call>"
        turns.append((roles[message["from"]], content))
    return turns

def tokenize_conversation(example):
    input_ids, labels = [], []
    for text, is_assistant_turn in format_conversation(json.loads(example["tools"]), glaive_to_turns(example)):
        ids = tokenizer(text, add_special_tokens=False).input_ids
        input_ids += ids
        labels += ids if is_assistant_turn else [-100] * len(ids)## FILL THE GAP: the model only learns to generate the assistant turns
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids), "labels": labels}

tool_data = tool_data.filter(lambda example: example["tools"] != "[]")  # keep the conversations with tools
tool_data = tool_data.shuffle(seed=42).map(tokenize_conversation, remove_columns=tool_data.column_names)
print(tool_data)

example = tool_data[0]
print("\n----- a training conversation:\n")
print(tokenizer.decode(example["input_ids"]))
print("\n----- what the model is trained to generate (labels != -100):\n")
print(tokenizer.decode([token for token in example["labels"] if token != -100]))

README.md:   0%|          | 0.00/524 [00:00<?, ?B/s]

glaive_toolcall_en_1k.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/670 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 670
})

----- a training conversation:

<tools>: [{"name": "calculate_tax", "description": "Calculate the tax amount", "parameters": {"type": "object", "properties": {"subtotal": {"type": "number", "description": "The subtotal amount"}, "tax_rate": {"type": "number", "description": "The tax rate in decimal form"}}, "required": ["subtotal", "tax_rate"]}}]
<human>: Hi, I need to calculate the tax on a purchase I made. The subtotal was $200 and the tax rate is 0.07. Can you help me with that?
<assistant>: <call>{"name": "calculate_tax", "arguments": {"subtotal": 200, "tax_rate": 0.07}}</call><|endoftext|>
<result>: {"tax_amount": 14}
<assistant>: The tax amount for your purchase is $14.<|endoftext|>


----- what the model is trained to generate (labels != -100):

 <call>{"name": "calculate_tax", "arguments": {"subtotal": 200, "tax_rate": 0.07}}</call><|endoftext|> The tax amount for your purchase is $14.<|end

#### <b>Finetuning:</b>

We add a new LoRA adapter (with the same configuration as in the first part) to the base model and train it with the same hyperparameters, for 50 steps, i.e. 400 conversations (about 7 minutes on a T4). If you run out of time, load our finetuned adapter at the beginning of the next cell instead.

In [ ]:
tool_model = get_peft_model(tool_model, config)  # a new LoRA adapter, with the same configuration as before

tool_trainer = transformers.Trainer(
    model=tool_model,
    train_dataset=tool_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-4,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_steps=50,  # 50 x 8 = 400 conversations
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=0.01,
        logging_steps=10,
        output_dir="experiments_tools",
        report_to="none",
    ),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=tool_model),
)

tool_model.config.use_cache = False
tool_trainer.train()

Step,Training Loss
10,0.754837
20,0.308266
30,0.231827
40,0.243564
50,0.155126


TrainOutput(global_step=50, training_loss=0.3387240815162659, metrics={'train_runtime': 395.4845, 'train_samples_per_second': 1.011, 'train_steps_per_second': 0.126, 'total_flos': 360068006530560.0, 'train_loss': 0.3387240815162659, 'epoch': 0.5970149253731343})

#### <b>After finetuning:</b>

Let us ask the same questions again:

In [ ]:
## uncomment if you didn't have enough time to train: this downloads our finetuned adapters from Google Drive and loads the tool-calling one
# !gdown --folder "https://drive.google.com/drive/folders/1HFLCqJ-MWXm_X36NvpQYaA1_GxT4cQNW" -O .
# tool_model = AutoModelForCausalLM.from_pretrained(
#                     MODEL_NAME,
#                     device_map="auto",
#                     dtype=torch.float16,
#                     quantization_config=bnb_config,
#                 )
# tool_model = PeftModel.from_pretrained(tool_model, "lab2_tool_adapter")

for question in questions:
    chat_with_tools(tool_model, question)
    print()

<human>: What's the weather like in Paris right now?
<assistant>: <call>{"name": "get_weather", "arguments": {"city": "Paris"}}</call>
<result>: {"city": "Paris", "weather": "12°C, light rain"}
<assistant>: The weather in Paris is currently 12°C with light rain.

<human>: Should I take an umbrella in Abu Dhabi today?
<assistant>: <call>{"name": "get_weather", "arguments": {"city": "Abu Dhabi"}}</call>
<result>: {"city": "Abu Dhabi", "weather": "35°C, sunny"}
<assistant>: The current weather in Abu Dhabi is 35°C with sunny conditions. You can take an umbrella if you need.

<human>: Can you book a flight to Paris for me?
<assistant>: I'm sorry, but I don't have the capability to book flights. My current function allows me to get the current weather in a given city. If you need help with that, feel free to ask!



<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 5: </b><br>
In <i>tokenize_conversation</i>, the labels of the tool results are set to -100, and each assistant turn ends with the end-of-sequence token. Explain why both choices matter for tool calling.
<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid red"> </hr>
Answer 5: </b><br>

<hr style="border:10px solid red"> </hr>
</font></h4>

Indeed, the model makes a call to the tool, and it has to learn how to make this call, but it must not learn to generate the result that the tool returns. This is why it is relevant to label the tool result with -100. Otherwise, the model would learn to generate the tool results, it would learn that a call is followed by a result, and it would tend to directly predict the result itself, which would be completely wrong, since we assume that the tool provides information that the model is not able to predict, such as the weather. Moreover, since these results depend on the outside world and are unpredictable, trying to learn them would be noise that would waste the limited learning capacity we have (LoRA, 400 conversations), which should rather be focused on when to call a tool, how to write the call and how to use the result. Nevertheless, the model does read the result, since its `attention_mask` is 1 even though its labels are -100. Therefore, when learning to generate the final answer, which does have labels, the model does so conditioned on the result, so it learns to use it without tending to invent it.

The importance of the EOS token in tool use is twofold. On the one hand, the model must be able to stop after calling a tool, in order to hand over the turn to the application, which runs the tool, appends the result to the conversation and launches the generation again. Otherwise, it would most likely keep generating text, trying to produce the result that is supposed to come from the tool, and would most likely write an incorrect answer. On the other hand, once the result has been received, it must also be able to write a message and end it, without generating text until reaching the maximum number of tokens set for the answer, nor inventing new conversation turns (`<human>:`, `<assistant>:`), as we observed with the base model. The model learns to generate this EOS token because it belongs to the assistant segment marked as `True` in `format_conversation`, and therefore it has a label (and not -100).

Both choices therefore protect the same principle: the results must come from the real tool and not from the model. The -100 prevents it from learning to write results, and the EOS token makes it stop before writing any.

We can observe this experimentally: before finetuning, the model never calls the tool and directly invents the weather (18 °C in Paris, 28 °C in Abu Dhabi), even generating an invented `<user>:` turn. After finetuning, it writes the call, stops, and uses the real result (12 °C and light rain in Paris, 35 °C and sunny in Abu Dhabi). Moreover, when asked to book a flight, it answers that it does not have this capability instead of claiming that it can do it.

### <b>Bonus: finetuning on another language (Catalan)</b>

We repeat the summarization finetuning in **Catalan**, using [CaSum](https://huggingface.co/datasets/projecte-aina/casum) (Projecte AINA, CC BY 4.0), a dataset of Catalan news articles paired with a short headline-style summary. To keep the same scale as the French experiment, we only download its validation file (10,000 examples, 22 MB instead of 487 MB for the whole dataset) and keep 5,000 of them.

We use the same chat template, with the instruction translated into Catalan (`Resumeix l’article següent:`), the same LoRA configuration and the same training hyperparameters. To compare fairly, we start from a fresh copy of the base model, compute the held-out loss before and after finetuning, and compare the generated summary of a held-out article with its reference summary.

In [ ]:
# Setup for the Catalan experiment (same values as in the previous parts)
from transformers import DataCollatorForSeq2Seq

device = "cuda:0"
MAX_LENGTH = 2000

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj",
                    "up_proj", "down_proj", "out_proj", "in_proj_qkv",
                    "in_proj_z", "in_proj_b", "in_proj_a"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id
generation_config.do_sample = True

In [ ]:
import gc

# free the GPU memory used by the previous parts (the merged 16-bit model is the largest one)
for name in ["merged_model", "base_model", "tool_model", "tool_trainer", "trainer"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

ca_data = load_dataset("projecte-aina/casum", data_files={"validation": "data/valid.jsonl"}, split="validation")
ca_data = ca_data.shuffle(seed=42).select(range(5000))
pd.DataFrame(ca_data[:5])

README.md:   0%|          | 0.00/6.11k [00:00<?, ?B/s]

data/valid.jsonl: reconstructing file:   0%|          |  0.00B / 22.4MB            

data/valid.jsonl: downloading bytes:           |  0.00B            

Generating validation split: 0 examples [00:00, ? examples/s]

,text,summary
0,"La farmacèutica Almirall, enfocada en la salut...",La farmacèutica Almirall entra a l'Íbex 35
1,"El president de la Generalitat, Quim Torra, vo...",Torra col·locarà una placa al Palau recordant ...
2,"El coordinador general del PPC, Xavier García ...",Albiol acusa Junqueras de “forçar” perquè la C...
3,El Patronat de l'Hospital Comarcal d'Amposta h...,L'Hospital Comarcal d'Amposta inverteix mig mi...
4,Els estudiants han tallat aquest dimecres al m...,ACTUALIZACIÓ:Talls als accessos a la UAB duran...


In [ ]:
CA_INSTRUCTION = "Resumeix l’article següent:"

def generate_prompt_ca(data_point):
    return f"<human>: {CA_INSTRUCTION}\n{data_point['text']}\n<assistant>: {data_point['summary']}"

def generate_and_tokenize_prompt_ca(data_point):
    full_prompt = generate_prompt_ca(data_point) + tokenizer.eos_token
    tokenized_full_prompt = tokenizer(full_prompt, return_tensors='pt')
    labels = tokenized_full_prompt.input_ids.clone()
    prompt = full_prompt[:full_prompt.find("<assistant>")] + "<assistant>:"
    end_prompt_idx = tokenizer(prompt, return_tensors='pt').input_ids.shape[-1]
    labels[:, :end_prompt_idx] = -100  # only the summary (and the EOS token) is learned
    return {
        'input_ids': tokenized_full_prompt.input_ids.flatten(),
        'labels': labels.flatten(),
        'attention_mask': tokenized_full_prompt.attention_mask.flatten(),
    }

ca_dataset = ca_data.map(generate_and_tokenize_prompt_ca, remove_columns=ca_data.column_names)
keep = [i for i, ids in enumerate(ca_dataset["input_ids"]) if len(ids) <= MAX_LENGTH]
ca_dataset = ca_dataset.select(keep)
ca_raw = ca_data.select(keep)  # the corresponding raw articles, to test the generation later

ca_split = ca_dataset.train_test_split(test_size=50, seed=42)
ca_train_data, ca_eval_data = ca_split["train"], ca_split["test"]
print(ca_train_data)
print(ca_eval_data)

# sanity check: what the model is trained to generate on one example
print("\nLearned part of an example:", tokenizer.decode([t for t in ca_train_data["labels"][0] if t != -100]))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 4887
})
Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 50
})

Learned part of an example:  Munté, sobre la citació de Rajoy: "És important que en qüestions tan sensibles es puguin aclarir els fets"<|endoftext|>


In [ ]:
def generate_summary_ca(model, article):
    prompt = f"<human>: {CA_INSTRUCTION}\n{article}\n<assistant>:"
    encoding = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=encoding.input_ids,
            attention_mask=encoding.attention_mask,
            generation_config=generation_config,
        )
    return tokenizer.decode(outputs[0][encoding.input_ids.shape[1]:], skip_special_tokens=True).strip()

# a held-out article: same split (same seed, same size) applied to the raw articles
ca_raw_eval = ca_raw.train_test_split(test_size=50, seed=42)["test"]
test_article, test_reference = ca_raw_eval[0]["text"], ca_raw_eval[0]["summary"]
print("ARTICLE:\n", test_article[:1000], "...\n")
print("REFERENCE SUMMARY:\n", test_reference)

# fresh base model, without any adapter
ca_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.float16,
    quantization_config=bnb_config,
)
torch.manual_seed(0)
print("\nBEFORE FINETUNING:\n", generate_summary_ca(ca_model, test_article))

ARTICLE:
 El secretari de Política Municipal del PSC i candidat socialista a Barcelona, Jaume Collboni, ha llençat un avís al president de la Generalitat, Quim Torra, i als partits independentistes que reclamen un gest del govern espanyol en favor del dret a l'autodeterminació per poder aprovar els pressupostos de Pedro Sánchez. "Quan plantegen propostes que van més enllà del nostre marc legal, que van més enllà de la llei, saben que no podem passar per aquí", ha advertit en roda de premsa. A més, el socialista ha defensat els comptes presentats aquest dilluns a Madrid i ha remarcat que si el sobiranisme els hi tanca la porta, estarà barrant el pas a "totes les millores que suposen" per als catalans i els espanyols. En aquest sentit, Collboni ha assegurat que aquells partits "que diuen no als pressupostos, diuen no a la millora de les condicions de vida dels ciutadans d'aquest país, diuen no a millora de la dependència, diuen no a polítiques d'habitatge, diuen no a la millora de les pe

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.



BEFORE FINETUNING:
 1. El secretari de Política Municipal del PSC i candidat socialista a Barcelona, Jaume Collboni, ha llençat un avís al president de la Generalitat, Quim Torra, i als partits independentistes que reclamen un gest del govern espanyol en favor del dret a l'autodeterminació per poder aprovar els pressupostos de Pedro Sánchez.
2. A més, el socialista ha defensat els comptes presentats aquest dilluns a Madrid i ha remarcat que si el sobiranisme els hi tanca la porta, estarà barrant el pas a "totes les millores que suposen" per als catalans i els espanyols.
3. En aquest sentit, Collboni ha assegurat que aquells partits "que diuen no als pressupostos, diuen no a la millora de les condicions de vida dels ciutadans d'aquest país,


In [ ]:
ca_model = get_peft_model(ca_model, config)  # a new LoRA adapter, with the same configuration as before
print_trainable_parameters(ca_model)

ca_trainer = transformers.Trainer(
    model=ca_model,
    train_dataset=ca_train_data,
    eval_dataset=ca_eval_data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-4,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_steps=100,  # 100 x 8 = 800 articles, as for French
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        per_device_eval_batch_size=1,
        save_total_limit=1,
        output_dir="experiments_ca",
        report_to="none",
    ),
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=ca_model),
)

ca_model.config.use_cache = False
ca_loss_before = ca_trainer.evaluate()["eval_loss"]
print("Held-out loss before finetuning:", ca_loss_before)
ca_trainer.train()
ca_loss_after = ca_trainer.evaluate()["eval_loss"]
print("Held-out loss before / after finetuning:", ca_loss_before, "/", ca_loss_after)

trainable params: 21645312 || all params: 774038336 || trainable%: 2.7964134324220344


Training Loss,Validation Loss,Step
No log,1.709789,0


Held-out loss before finetuning: 1.7097893953323364


Step,Training Loss,Validation Loss
25,1.326480,1.317861
50,1.289647,1.327961
75,1.185629,1.229330
100,1.082008,1.154793


Training Loss,Validation Loss,Step
1.082008,1.154793,100


Held-out loss before / after finetuning: 1.7097893953323364 / 1.1547925472259521


In [ ]:
ca_model.config.use_cache = True
print("REFERENCE SUMMARY:\n", test_reference)
for seed in [0, 1]:
    torch.manual_seed(seed)
    print(f"\nAFTER FINETUNING (seed {seed}):\n", generate_summary_ca(ca_model, test_article))

# training and held-out losses logged during training
pd.DataFrame([log for log in ca_trainer.state.log_history if "loss" in log or "eval_loss" in log])

REFERENCE SUMMARY:
 El PSC avisa Torra que "planteja propostes que van més enllà de la llei" en les seves exigències per votar el pressupost

AFTER FINETUNING (seed 0):
 Collboni reclama un gest del govern espanyol en favor del dret a l'autodeterminació per aprovar els pressupostos

AFTER FINETUNING (seed 1):
 Collboni reclama un gest del govern espanyol en favor del dret a l'autodeterminació per aprovar els pressupostos


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_model_preparation_time,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,1.748352,11.051850,0.000000,0.016370,10,NaN,NaN,NaN,NaN,NaN
1,1.326480,5.434882,0.000490,0.032740,20,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,0.040925,25,1.317861,0.0358,14.6535,3.412,3.412
3,1.352002,5.389727,0.000456,0.049110,30,NaN,NaN,NaN,NaN,NaN
4,1.278783,5.129808,0.000401,0.065480,40,NaN,NaN,NaN,NaN,NaN
5,1.289647,6.074032,0.000332,0.081850,50,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,0.081850,50,1.327961,0.0358,14.7600,3.388,3.388
7,1.344133,4.659947,0.000254,0.098220,60,NaN,NaN,NaN,NaN,NaN
8,1.185629,4.032883,0.000176,0.114590,70,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,0.122775,75,1.229330,0.0358,24.5720,2.035,2.035


**Results:** The validation loss decreases from 1.710 to 1.155 (−32%) after 100 steps (800 articles), and it decreases steadily along training (1.318, 1.328, 1.229 and 1.155 at steps 25, 50, 75 and 100), while the training loss goes from 1.75 to 1.08, so there are no signs of overfitting.

Qualitatively, the base model does not summarize the held-out article: it copies its first sentences verbatim as a numbered list and does not stop until the token limit. After finetuning, it writes a single headline-style sentence in Catalan and stops by itself, so it has learned the format of the task and to generate the EOS token. However, the content is not faithful: the generated summary attributes to Collboni the demand of a gesture towards self-determination, whereas in the article this demand comes from the pro-independence parties and Collboni warns against it, as the reference summary states ("El PSC avisa Torra..."). Note also that the generations with seeds 0 and 1 are identical: after finetuning the distribution is so peaked that, with $T=0.7$ and top-$p=0.7$, the nucleus often contains a single token and sampling becomes nearly deterministic.

These observations match the French experiment, where the finetuned model also produced a short summary that stops by itself but with an inaccurate content (the 14% pay gap presented as a consequence of the directive). In both cases, a small 4-bit model trained on 800 examples (16% of an epoch) learns the style of the summaries well, but not to summarize faithfully. Quantitatively, the improvement is much larger in Catalan (held-out loss −32%, from 1.710 to 1.155) than in French (−6.5%, from 2.331 to 2.179), although the absolute values are not directly comparable, since the datasets have different summary styles and lengths. The short, headline-like CaSum summaries seem easier to learn in 100 steps than the longer OrangeSum ones.


*Reference:* projecte-aina, *CaSum: a Catalan summarization dataset*. [huggingface.co/datasets/projecte-aina/casum](https://huggingface.co/datasets/projecte-aina/casum)

To go further:
- Check **vLLM** for fast batch inference.
- Check **DDP**, **FSDP** and **DeepSpeed** for distributed training with Hugging Face transformers.
- Check **Unsloth** for faster training.
- Check **Ollama** for a chat interface.
- Test **multi-turn** and **few-shot learning**.
- Compare with the post-trained chat model `Qwen/Qwen3.5-0.8B`.
- Check **Megatron, Nanotron, etc.** for distributed **pre-training** on big clusters.
- Check **LLaMA-Factory** (https://github.com/hiyouga/LLaMA-Factory) for **Finetuning**.